In [2]:
from google.colab import files

print("Upload the ZIP containing your 15 videos...")

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one ZIP file.")

zip_name = list(uploaded.keys())[0]

if not zip_name.lower().endswith(".zip"):
    raise ValueError("The uploaded file must be a .zip file.")

print("Uploaded:", zip_name)

Upload the ZIP containing your 15 videos...


Saving real-life-footage.zip to real-life-footage.zip
Uploaded: real-life-footage.zip


In [4]:
# ============================================================
# CELL 2: EXTRACT AND VERIFY VIDEOS
# ============================================================

import zipfile
import os
import shutil

VIDEO_DIR = "/content/videos"

# ------------------------------------------------------------
# Remove previous extraction if it exists
# ------------------------------------------------------------

if os.path.exists(VIDEO_DIR):
    shutil.rmtree(VIDEO_DIR)

os.makedirs(VIDEO_DIR)

# ------------------------------------------------------------
# Extract ZIP
# ------------------------------------------------------------

zip_path = f"/content/{zip_name}"

print("Extracting ZIP...")
print("This may take some time depending on the size.")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(VIDEO_DIR)

print("Extraction complete!")

# ------------------------------------------------------------
# Find videos recursively
# ------------------------------------------------------------

VIDEO_EXTENSIONS = (
    ".mp4",
    ".avi",
    ".mov",
    ".mkv",
    ".webm"
)

videos = []

for root, dirs, files_in_dir in os.walk(VIDEO_DIR):

    for filename in files_in_dir:

        if filename.lower().endswith(VIDEO_EXTENSIONS):

            videos.append(
                os.path.join(root, filename)
            )

videos.sort()

# ------------------------------------------------------------
# Verify number of videos
# ------------------------------------------------------------

print("\n==========================================")
print("VIDEO VERIFICATION")
print("==========================================")

print("Videos found:", len(videos))

for i, video in enumerate(videos, start=1):

    print(
        f"{i:2d}. "
        f"{os.path.basename(video)}"
    )

if len(videos) != 15:

    raise ValueError(
        f"\nExpected exactly 15 videos, "
        f"but found {len(videos)}.\n"
        f"Check the ZIP contents before continuing."
    )

print("\n✓ Exactly 15 videos found.")

Extracting ZIP...
This may take some time depending on the size.
Extraction complete!

VIDEO VERIFICATION
Videos found: 15
 1. lv_0_20260911182552.mp4
 2. lv_0_20260911182821.mp4
 3. lv_0_20260911182928.mp4
 4. lv_0_20260911183028.mp4
 5. lv_0_20260911183122.mp4
 6. lv_0_20260911183220.mp4
 7. lv_0_20260911183257.mp4
 8. lv_0_20260911183329.mp4
 9. lv_0_20260911183408.mp4
10. lv_0_20260911183435.mp4
11. lv_0_20260911183459.mp4
12. lv_0_20260911183518.mp4
13. lv_0_20260911183533.mp4
14. lv_0_20260911183548.mp4
15. lv_0_20260911183559.mp4

✓ Exactly 15 videos found.


In [5]:
# ============================================================
# CELL 3: SAMPLE 250 DIVERSE FRAMES
# ============================================================

import cv2
import os
import random
import shutil
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

TOTAL_SAMPLES = 250

# Minimum separation between two selected frames
MIN_FRAME_DISTANCE = 100

# Makes the sampling reproducible
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

# Output folder
OUTPUT_DIR = "/content/roboflow_frames"

# ------------------------------------------------------------
# Clean old output
# ------------------------------------------------------------

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(OUTPUT_DIR)

# ============================================================
# ANALYZE VIDEOS
# ============================================================

video_info = []

print("==========================================")
print("ANALYZING VIDEOS")
print("==========================================")

for video_path in videos:

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video:\n{video_path}"
        )

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = cap.get(cv2.CAP_PROP_FPS)

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    cap.release()

    # --------------------------------------------------------
    # Verify video information
    # --------------------------------------------------------

    if total_frames <= 0:

        raise RuntimeError(
            f"Could not determine frame count:\n{video_path}"
        )

    if fps <= 0:

        raise RuntimeError(
            f"Could not determine FPS:\n{video_path}"
        )

    name = os.path.splitext(
        os.path.basename(video_path)
    )[0]

    duration = total_frames / fps

    info = {
        "path": video_path,
        "name": name,
        "frames": total_frames,
        "fps": fps,
        "width": width,
        "height": height,
        "duration": duration
    }

    video_info.append(info)

    print(
        f"{name:20s} | "
        f"{width}x{height} | "
        f"{fps:.2f} FPS | "
        f"{total_frames:,} frames | "
        f"{duration:.1f} sec"
    )

# ============================================================
# DISTRIBUTE 250 SAMPLES
# ============================================================

num_videos = len(video_info)

base_samples = TOTAL_SAMPLES // num_videos
extra_samples = TOTAL_SAMPLES % num_videos

print("\n==========================================")
print("SAMPLING DISTRIBUTION")
print("==========================================")

print(
    f"Total samples: {TOTAL_SAMPLES}"
)

print(
    f"Videos: {num_videos}"
)

print(
    f"Base samples/video: {base_samples}"
)

print(
    f"Extra samples: {extra_samples}"
)

# ============================================================
# SELECT FRAME NUMBERS
# ============================================================

samples = []

for video_index, info in enumerate(video_info):

    total_frames = info["frames"]

    # --------------------------------------------------------
    # 16 or 17 frames per video
    # --------------------------------------------------------

    num_samples = base_samples

    if video_index < extra_samples:
        num_samples += 1

    print(
        f"\n{info['name']} "
        f"→ {num_samples} frames"
    )

    selected = []

    # --------------------------------------------------------
    # Divide the video into temporal sections
    # --------------------------------------------------------

    section_size = (
        total_frames / num_samples
    )

    for section in range(num_samples):

        start = int(
            section * section_size
        )

        end = int(
            (section + 1) * section_size
        ) - 1

        # Safety
        start = max(
            0,
            start
        )

        end = min(
            total_frames - 1,
            end
        )

        if end < start:
            end = start

        # ----------------------------------------------------
        # Generate random candidates
        # ----------------------------------------------------

        candidates = list(
            range(start, end + 1)
        )

        random.shuffle(candidates)

        chosen = None

        # ----------------------------------------------------
        # Find candidate sufficiently far from
        # already selected frames
        # ----------------------------------------------------

        for candidate in candidates:

            is_far_enough = all(
                abs(candidate - previous)
                >= MIN_FRAME_DISTANCE
                for previous in selected
            )

            if is_far_enough:

                chosen = candidate
                break

        # ----------------------------------------------------
        # Fallback
        # ----------------------------------------------------

        if chosen is None:

            chosen = random.randint(
                start,
                end
            )

        selected.append(chosen)

    # --------------------------------------------------------
    # Sort frames chronologically
    # --------------------------------------------------------

    selected.sort()

    # --------------------------------------------------------
    # Save sample information
    # --------------------------------------------------------

    for frame_number in selected:

        samples.append({
            "video": info["name"],
            "path": info["path"],
            "frame": frame_number,
            "fps": info["fps"]
        })

# ============================================================
# SHUFFLE FINAL DATASET
# ============================================================

random.shuffle(samples)

print("\n==========================================")
print("FRAME SELECTION COMPLETE")
print("==========================================")

print(
    "Total frames selected:",
    len(samples)
)

# ============================================================
# EXTRACT FRAMES
# ============================================================

captures = {}

successful = []

print("\n==========================================")
print("EXTRACTING FRAMES")
print("==========================================")

for counter, sample in enumerate(
    samples,
    start=1
):

    video_path = sample["path"]

    # --------------------------------------------------------
    # Open video once
    # --------------------------------------------------------

    if video_path not in captures:

        cap = cv2.VideoCapture(
            video_path
        )

        if not cap.isOpened():

            print(
                "WARNING: Could not open:",
                video_path
            )

            continue

        captures[video_path] = cap

    cap = captures[video_path]

    frame_number = sample["frame"]

    # --------------------------------------------------------
    # Move to selected frame
    # --------------------------------------------------------

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        frame_number
    )

    success, frame = cap.read()

    if not success or frame is None:

        print(
            f"WARNING: Failed to read "
            f"{sample['video']} "
            f"frame {frame_number}"
        )

        continue

    # --------------------------------------------------------
    # Filename
    # --------------------------------------------------------

    filename = (
        f"{sample['video']}"
        f"_frame_{frame_number:07d}.jpg"
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    # --------------------------------------------------------
    # Save JPEG
    # --------------------------------------------------------

    saved = cv2.imwrite(
        output_path,
        frame,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            95
        ]
    )

    if not saved:

        print(
            f"WARNING: Could not save "
            f"{filename}"
        )

        continue

    successful.append({
        "filename": filename,
        "video": sample["video"],
        "frame": frame_number,
        "time_seconds": (
            frame_number /
            sample["fps"]
        )
    })

    # Progress
    if counter % 25 == 0:

        print(
            f"Processed "
            f"{counter}/{len(samples)}"
        )

# ============================================================
# RELEASE ALL VIDEOS
# ============================================================

for cap in captures.values():
    cap.release()

# ============================================================
# CREATE CSV
# ============================================================

df = pd.DataFrame(
    successful
)

csv_path = "/content/sampled_frames.csv"

df.to_csv(
    csv_path,
    index=False
)

# ============================================================
# FINAL VERIFICATION
# ============================================================

actual_files = [
    f
    for f in os.listdir(OUTPUT_DIR)
    if f.lower().endswith(".jpg")
]

print("\n==========================================")
print("SAMPLING FINISHED")
print("==========================================")

print(
    "Requested frames:",
    TOTAL_SAMPLES
)

print(
    "Successfully created:",
    len(actual_files)
)

print(
    "Output directory:",
    OUTPUT_DIR
)

print(
    "CSV:",
    csv_path
)

# ------------------------------------------------------------
# Frames per video
# ------------------------------------------------------------

print("\nFrames per video:")

print(
    df.groupby("video")
      .size()
      .to_string()
)

# ============================================================
# CREATE ZIP
# ============================================================

print("\nCreating ZIP...")

zip_path = shutil.make_archive(
    "/content/roboflow_frames",
    "zip",
    OUTPUT_DIR
)

print("\nZIP created:")
print(zip_path)

ANALYZING VIDEOS
lv_0_20260911182552  | 1280x720 | 30.00 FPS | 1,796 frames | 59.9 sec
lv_0_20260911182821  | 1280x720 | 30.00 FPS | 69 frames | 2.3 sec
lv_0_20260911182928  | 1280x720 | 30.00 FPS | 159 frames | 5.3 sec
lv_0_20260911183028  | 1280x720 | 30.00 FPS | 93 frames | 3.1 sec
lv_0_20260911183122  | 1280x720 | 30.00 FPS | 34 frames | 1.1 sec
lv_0_20260911183220  | 1280x720 | 30.00 FPS | 110 frames | 3.7 sec
lv_0_20260911183257  | 1280x720 | 30.00 FPS | 54 frames | 1.8 sec
lv_0_20260911183329  | 1280x720 | 30.00 FPS | 282 frames | 9.4 sec
lv_0_20260911183408  | 1280x720 | 30.00 FPS | 69 frames | 2.3 sec
lv_0_20260911183435  | 1280x720 | 30.00 FPS | 90 frames | 3.0 sec
lv_0_20260911183459  | 1280x720 | 30.00 FPS | 88 frames | 2.9 sec
lv_0_20260911183518  | 1280x720 | 30.00 FPS | 72 frames | 2.4 sec
lv_0_20260911183533  | 1280x720 | 30.00 FPS | 135 frames | 4.5 sec
lv_0_20260911183548  | 1280x720 | 30.00 FPS | 408 frames | 13.6 sec
lv_0_20260911183559  | 1280x720 | 30.00 FPS | 137

In [6]:
# ============================================================
# CELL 4: VERIFY AND DOWNLOAD
# ============================================================

import zipfile
import os

zip_path = "/content/roboflow_frames.zip"

# ------------------------------------------------------------
# Verify ZIP exists
# ------------------------------------------------------------

if not os.path.exists(zip_path):

    raise FileNotFoundError(
        "roboflow_frames.zip was not created."
    )

# ------------------------------------------------------------
# Inspect ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(
    zip_path,
    "r"
) as z:

    jpg_files = [
        name
        for name in z.namelist()
        if name.lower().endswith(".jpg")
    ]

print("==========================================")
print("ZIP VERIFICATION")
print("==========================================")

print(
    "Images inside ZIP:",
    len(jpg_files)
)

if len(jpg_files) != 250:

    raise ValueError(
        f"Expected 250 images, "
        f"but ZIP contains {len(jpg_files)}."
    )

print("✓ ZIP contains exactly 250 images.")

# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

print("\nStarting download...")

from google.colab import files

files.download(zip_path)

ZIP VERIFICATION
Images inside ZIP: 250
✓ ZIP contains exactly 250 images.

Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>